In [117]:
import pandas as pd
import numpy as np
from scipy.stats import norm
import os

In [118]:
raw_ihs_poverty_2010 = pd.read_csv(r'd:\GG\source\householdpoverty_10.CSV')
raw_ihs_weight_2010 = pd.read_csv(r'd:\GG\source\householdweight_10.CSV')
raw_ihs_weight_2015 = pd.read_csv(r'd:\GG\source\householdweight_15.CSV')
raw_ihs_poverty_2015 = pd.read_csv(r'd:\GG\source\householdpoverty_15.CSV')
raw_dhs_2020 = pd.read_stata(r'd:\GG\source\household_19_20.DTA')
raw_findex_2021 = pd.read_csv(r'd:\GG\source\connectivity_21.csv')
raw_findex_2024 = pd.read_csv(r'd:\GG\source\connectivity_24.csv')

In [119]:
# aggregate economic rank
# 2010
df_2010 = pd.DataFrame()
df_2010['hid'] = raw_ihs_poverty_2010['hid']
df_2010['survey_year'] = 2010
df_2010['data_source'] = 'IHS'
df_2010['lga'] = raw_ihs_poverty_2010['lga']

weight_lookup = raw_ihs_weight_2010[['hid', 'weightslga']].drop_duplicates(subset=['hid'])
df_2010 = df_2010.merge(weight_lookup, on='hid', how='left')
df_2010['weightslga'] = df_2010['weightslga'].fillna(np.nan)

poverty_map = raw_ihs_poverty_2010.drop_duplicates('hid').set_index('hid')['s11q2']
df_2010['hh_income'] = df_2010['hid'].astype(int).map(poverty_map)

# 2015
df_2015 = pd.DataFrame()
df_2015['hh_id'] = raw_ihs_poverty_2015['hid']
df_2015['survey_year'] = 2015
df_2015['data_source'] = 'IHS'
df_2015['eanum'] = raw_ihs_poverty_2015['eanum']

weight_lookup = raw_ihs_weight_2015[['eanum', 'hhweight']].drop_duplicates(subset=['eanum'])
df_2015 = df_2015.merge(weight_lookup, on='eanum', how='left')
df_2015['hh_weight'] = df_2015['hhweight'].fillna(np.nan)

poverty_map = raw_ihs_poverty_2015.drop_duplicates('hid').set_index('hid')['s13q3']
df_2015['hh_income'] = df_2015['hh_id'].astype(int).map(poverty_map)

# 2020
df_2020 = pd.DataFrame()
df_2020['hh_id'] = raw_dhs_2020['hhid']
df_2020['survey_year'] = raw_dhs_2020['hv007']
df_2020['hh_income'] = raw_dhs_2020['hv270']

df_2020['data_source'] = 'DHS'
df_2020['weight'] = raw_dhs_2020['hv005']

quintile_labels = {'poorest': 1, 'poorer': 2, 'middle': 3, 'richer': 4, 'richest': 5}
df_2020['hh_income'] = df_2020['hh_income'].map(quintile_labels)

# 2021
df_2021 = pd.DataFrame()
df_2021['hh_id'] = raw_findex_2021.index.map(lambda x: f"findex_21_{x}")
df_2021['survey_year'] = 2021
df_2021['data_source'] = 'Findex'
df_2021['hh_income'] = raw_findex_2021['inc_q']
df_2021['weight'] = (raw_findex_2021['wgt']).fillna(NA)

# 2024
df_2024 = pd.DataFrame()
df_2024['hh_id'] = raw_findex_2024.index.map(lambda x: f"findex_24_{x}")
df_2024['survey_year'] = 2024
df_2024['data_source'] = 'Findex'
df_2024['hh_income'] = raw_findex_2024['inc_q']
df_2024['weight'] = (raw_findex_2024['wgt']).fillna(NA)

In [120]:
# econ rank construct
df_2010['hh_econ_rank'] = (df_2010.sort_values('hh_income')['weightslga'].cumsum() - 0.5 * df_2010['weightslga']) / df_2010['weightslga'].sum() * 100
df_2015['hh_econ_rank'] = (df_2015.sort_values('hh_income')['hh_weight'].cumsum() - 0.5 * df_2015['hh_weight']) / df_2015['hh_weight'].sum() * 100
df_2020['hh_econ_rank'] = (df_2020.sort_values('hh_income')['weight'].cumsum() - 0.5 * df_2020['weight']) / df_2020['weight'].sum() * 100
df_2021['hh_econ_rank'] = (df_2021.sort_values('hh_income')['weight'].cumsum() - 0.5 * df_2021['weight']) / df_2021['weight'].sum() * 100
df_2024['hh_econ_rank'] = (df_2024.sort_values('hh_income')['weight'].cumsum() - 0.5 * df_2024['weight']) / df_2024['weight'].sum() * 100

In [121]:
# concat all
df_2010.rename(columns={'hid': 'hh_id', 'weightslga': 'weight'}, inplace=True)
df_2015.rename(columns={'hh_weight': 'weight'}, inplace=True)

target_cols = ['hh_id', 'survey_year', 'data_source', 'hh_income', 'weight', 'hh_econ_rank']

df_2010 = df_2010[target_cols]
df_2015 = df_2015[target_cols]
df_2020 = df_2020[target_cols]
df_2021 = df_2021[target_cols]
df_2024 = df_2024[target_cols]

dfs = [df_2010, df_2015, df_2020, df_2021, df_2024]
for df in dfs:
    df['hh_id'] = df['hh_id'].astype(str)

df_panel = pd.concat(dfs, ignore_index=True)
df_panel = df_panel.dropna(subset=['hh_id', 'hh_income', 'weight'])

df_panel['unique_id'] = (
    df_panel['data_source'] + '_' + 
    df_panel['survey_year'].astype(str) + '_' + 
    df_panel['hh_id']
)

df_panel.set_index('unique_id', inplace=True)

display(df_panel.head())

,hh_id,survey_year,data_source,hh_income,weight,hh_econ_rank
unique_id,,,,,,
IHS_2010_1101101000110003101,1101101000110003101,2010,IHS,2.0,0.59,51.423247
IHS_2010_1101101000110003102,1101101000110003102,2010,IHS,1.0,0.59,0.620941
IHS_2010_1101101000110003103,1101101000110003103,2010,IHS,2.0,0.59,7.880949
IHS_2010_1101101000110003104,1101101000110003104,2010,IHS,1.0,0.59,7.723077
IHS_2010_1101101000110003105,1101101000110003105,2010,IHS,2.0,0.59,7.893270
